# Lab 03: Your First Real Run, Classical KD and the Capacity Gap

**Tier 2 lab.** This is the first notebook in the course where a training loop executes. It has
the three-part shape every Tier 2 lab uses:

- **Part A, pre-flight.** Executed and asserted on any machine, no GPU needed. Config identity,
  memory arithmetic, loss verification, data preparation. If Part A passes, the run cannot fail
  for a *silent* reason.
- **Part B, the run.** The actual training, gated behind one flag. It was **not** executed
  during the build of this course, because a green checkmark produced on build hardware would be
  a false assurance about yours. Flip the flag on the training box.
- **Part C, the verdict.** Expected ranges, failure signatures, and the judgment you write.

```python
RUN_TRAINING = False   # flip to True on the training box
```

**The question.** Two terms first, because the whole lab turns on them. A *hard label* is the
ordinary supervision signal: one correct token id per position, which is the same as a
probability distribution with 1.0 on the right token and 0 everywhere else. A *soft target* is
the teacher's full probability distribution over the vocabulary at that position: maybe 0.6 on
the right token, 0.2 on a plausible alternative, tiny amounts spread over everything else.
Hinton, Vinyals & Dean (2015) claimed a student learns more from a teacher's soft targets than
from hard labels. The reason is in the wrong answers: the relative probabilities the teacher
assigns to *incorrect* tokens (the paper's nickname for this is "dark knowledge") encode how
the teacher generalises. If the teacher puts a thousand times more probability on "cat" than
on "car" when the answer is "dog", that similarity structure is information a hard label does
not contain, and the student can absorb it. This lab tests that claim on a real model pair,
then probes its best-known failure: the **capacity gap**, which is the finding that a teacher
too far above the student in size can distill *worse* than a smaller teacher, because the
student cannot represent the function the large teacher is handing it (Müller et al. on label
smoothing killing dark knowledge; Beyer et al. on patience being worth more than teacher size).

**The arms.** An *arm* is one configuration in a controlled comparison; the word comes from
clinical trials, where each group of patients is an arm of the study. Five short runs, one
variable at a time:

| arm | teacher | student | α (soft weight) | tests |
|---|---|---|---|---|
| `hard` | none | 360M | 0.0 | the baseline KD must beat |
| `mixed` | 1.7B | 360M | 0.5 | the standard recipe |
| `soft` | 1.7B | 360M | 1.0 | pure distillation |
| `gap-small` | 360M | 135M | 0.5 | modest gap (2.7×) |
| `gap-large` | 1.7B | 135M | 0.5 | large gap (12.6×) |

α is the weight on the soft-target term of the loss: α=0 means the loss is all hard labels,
α=1 means it is all soft targets, and α=0.5 is an even mix. The first three arms answer "does
the soft target help?" The last two answer "does more teacher help?", and the honest
expectation, from the capacity-gap literature, is *not necessarily*.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, os, json, math
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

from kd_core import (hinton_kd_loss, kl_divergence, shift_for_next_token,
                     completion_mask_from_prompt_lens, top1_agreement,
                     mean_entropy, expected_calibration_error, masked_mean)
from kd_pipeline import (set_seed_everywhere, config_fingerprint, MemoryPlan,
                         full_ft_gb, infer_gb, RunManifest)

RUN_TRAINING = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False


## Part A · 1: Arms as configs, and configs as identities

Every run in this course is a config dict plus a seed, and its fingerprint goes into every
filename it produces. A *fingerprint* is a short hash of the config: a hash is a function that
turns any input into a short fixed-length string, built so that changing even one value in the
input produces a completely different string. That gives every distinct config a distinct name,
which means a checkpoint's filename tells you exactly which config produced it. This sounds
like bureaucracy until the first time you have seven checkpoints named `output_final_v2_fixed`
and no idea which loss produced which, which happens to everyone exactly once.

The assertion that matters: within each comparison group, arms must differ in **exactly one
key**. This is the core rule of an *ablation*, an experiment that isolates one factor by
changing it while holding everything else fixed. An ablation that varies two things measures
neither, because any difference in the outcome could be credited to either change.

In [2]:
BASE = dict(
    dataset="HuggingFaceTB/smol-smoltalk", n_train=4096, n_eval=256,
    seq_len=384, lr=3e-5, batch_size=8, grad_accum=4, max_steps=1500,
    warmup_steps=50, T=2.0, dtype="bfloat16",
)
ARMS = {
    "hard":      {**BASE, "teacher": None,                              "student": "HuggingFaceTB/SmolLM2-360M-Instruct", "alpha": 0.0},
    "mixed":     {**BASE, "teacher": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "student": "HuggingFaceTB/SmolLM2-360M-Instruct", "alpha": 0.5},
    "soft":      {**BASE, "teacher": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "student": "HuggingFaceTB/SmolLM2-360M-Instruct", "alpha": 1.0},
    "gap-small": {**BASE, "teacher": "HuggingFaceTB/SmolLM2-360M-Instruct", "student": "HuggingFaceTB/SmolLM2-135M-Instruct", "alpha": 0.5},
    "gap-large": {**BASE, "teacher": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "student": "HuggingFaceTB/SmolLM2-135M-Instruct", "alpha": 0.5},
}

def diff_keys(a: dict, b: dict) -> set:
    return {k for k in a.keys() | b.keys() if a.get(k) != b.get(k)}

# Group 1 (alpha question): same pair, alpha/teacher only.
assert diff_keys(ARMS["mixed"], ARMS["soft"]) == {"alpha"}
assert diff_keys(ARMS["hard"], ARMS["soft"]) == {"alpha", "teacher"}   # hard has no teacher
# Group 2 (gap question): same student and alpha, teacher only.
assert diff_keys(ARMS["gap-small"], ARMS["gap-large"]) == {"teacher"}

for name, cfg in ARMS.items():
    print(f"{name:>10}: {config_fingerprint({**cfg, 'seed': SEED})}")
print("\narm discipline verified: each comparison isolates one variable")

      hard: d9b15100c4bc
     mixed: ffa8b1247125
      soft: f1bbd5ff937b
 gap-small: d17687d05d2d
 gap-large: 7ca101f1cc63

arm discipline verified: each comparison isolates one variable


## Part A · 2: The memory plan, asserted before anything loads

The rule from the README: full fine-tuning costs about 16 bytes per parameter, and bf16
inference costs 2. Here is where the 16 comes from, per parameter: 2 bytes for the bf16 weight
itself, 2 bytes for its bf16 gradient, and 12 more bytes because the Adam optimizer keeps three
fp32 numbers per parameter (a 4-byte running average of the gradient, a 4-byte running average
of the squared gradient, and a 4-byte fp32 master copy of the weight so tiny updates are not
rounded away in bf16). 2 + 2 + 4 + 4 + 4 = 16. Inference stores only the 2-byte weight, hence
2. The plan below is for the *largest* arm, because if `gap-large` fits, everything fits. Note
what the headroom is for. Three costs are real and not covered by the per-parameter rule: the
KV cache (memory where the model stores the attention keys and values of tokens it has already
processed, so it does not recompute them for every new token), the activations (the
intermediate tensors of the forward pass, which scale with batch size and sequence length, not
parameter count), and CUDA allocator fragmentation (free memory carved into pieces too small
to satisfy the next allocation). The plan budgets slack for all three.

Do this arithmetic before every run you ever launch. The failure it prevents, an out-of-memory
crash forty minutes into a run after the caches warmed up, is the most demoralising failure in
the business, and the cheapest to avoid.

In [3]:
plan = (MemoryPlan(total_gb=128.0)
        .add("teacher 1.7B, bf16 inference", infer_gb(1.7))
        .add("student 360M, full fine-tune", full_ft_gb(0.36))
        .add("student activations (batch 8 x 384, est.)", 4.0)
        .add("teacher KV + activations (est.)", 3.0))
print(plan.table())
plan.assert_fits()

# And the arm this lab does NOT include, shown as a negative example:
too_big = (MemoryPlan(total_gb=128.0)
           .add("teacher 32B, bf16 inference", infer_gb(32))
           .add("student 4B, full fine-tune", full_ft_gb(4.0)))
assert not too_big.fits, "32B teacher + 4B full-FT student must NOT fit in 128 GB"
print(f"\nnegative control: 32B + 4B full-FT plans {too_big.planned_gb:.0f} GB -> refused, as it should be")

component                                    GB
teacher 1.7B, bf16 inference               3.40
student 360M, full fine-tune               5.76
student activations (batch 8 x 384, est.)     4.00
teacher KV + activations (est.)            3.00
-----------------------------------------------
planned                                   16.16
budget (after headroom)                  108.80
fits                                       True

negative control: 32B + 4B full-FT plans 128 GB -> refused, as it should be


## Part A · 3: The loss, re-verified at the door

Lab 01 proved `hinton_kd_loss` correct. This cell re-proves the three properties this lab's
arms depend on, because the loss is imported code and imported code changes:

1. `alpha=0` reduces to plain cross-entropy, which means the `hard` arm really is a no-teacher
   baseline: with the soft term weighted to zero, nothing the teacher says can reach the
   gradient.
2. `alpha=1` is pure soft KL, which means the `soft` arm has no hard-label term at all.
3. The `T²` factor keeps the soft-term gradient scale independent of `T` (the temperature, a
   divisor applied to logits before the softmax; higher T flattens both distributions). The
   flattening shrinks the soft term's gradients by roughly a factor of T², so multiplying the
   term by T² cancels the shrinkage. Because of that, the α weighting means the same thing at
   `T=2` as it would at `T=1`, and comparing arms across α is a fair comparison.

In [4]:
B, T_len, V = 2, 12, 128
s = torch.randn(B, T_len, V, requires_grad=True)
t = torch.randn(B, T_len, V)
labels = torch.randint(0, V, (B, T_len))
m = torch.ones(B, T_len, dtype=torch.bool)

ce = F.cross_entropy(s.reshape(-1, V), labels.reshape(-1))
assert torch.allclose(hinton_kd_loss(s, t, labels, m, T=2.0, alpha=0.0), ce, atol=1e-6)

pure_soft = kl_divergence(s, t, m, T=2.0, direction="forward", scale_by_T2=True)
assert torch.allclose(hinton_kd_loss(s, t, labels, m, T=2.0, alpha=1.0), pure_soft, atol=1e-6)

# T^2 scaling: soft-term gradients at T=2 and T=4 within a factor ~2, not ~4+.
def soft_grad_norm(T):
    s2 = s.detach().clone().requires_grad_(True)
    kl_divergence(s2, t, m, T=T, direction="forward", scale_by_T2=True).backward()
    return float(s2.grad.norm())
g2, g4 = soft_grad_norm(2.0), soft_grad_norm(4.0)
assert 0.3 < g2 / g4 < 3.0, "T^2 compensation keeps gradient scales comparable across T"
print(f"alpha endpoints verified; grad-norm ratio T=2 vs T=4: {g2/g4:.2f} (T^2 doing its job)")

alpha endpoints verified; grad-norm ratio T=2 vs T=4: 1.04 (T^2 doing its job)


## Part A · 4: Data, tokenized and mask-audited

The corpus is a slice of SmolTalk, which is the SmolLM2 family's own instruction data. That
choice is deliberate: distilling on prompts from the models' own training distribution
(in-distribution prompts) isolates the KD question from a separate domain-shift question,
which means any difference between arms comes from the loss, not from the data being
unfamiliar to one model or the other. Everything Lab 02 established gets applied here:
chat-template rendering, prompt lengths measured on the templated sequence, `<|endoftext|>`
padding so the real EOS stays supervised, and labels built as `input_ids` with `-100` outside
the completion mask (the boolean mask marking which positions belong to the assistant's
completion, the only positions the loss is allowed to train on).

The audit assertions at the end are the same four every dataset you ever build should pass.
This cell downloads a few thousand rows on first execution and writes the tensors to
`../data/lab03/` for Part B, and also for Labs 04 and 05, which reuse this exact corpus so
their results stay comparable with this lab's.

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
PAD_ID = tok.convert_tokens_to_ids("<|endoftext|>")
EOS_ID = tok.eos_token_id
CFG = ARMS["mixed"]

def encode(msgs, seq_len):
    prompt = tok.apply_chat_template(msgs[:-1], add_generation_prompt=True,
                                     tokenize=True, return_dict=False)
    comp = tok(msgs[-1]["content"], add_special_tokens=False)["input_ids"] + [EOS_ID]
    full = prompt + comp
    if not (8 < len(comp) and len(full) <= seq_len):
        return None
    return full, len(prompt)

stream = load_dataset(CFG["dataset"], split="train", streaming=True)
encoded, need, scanned = [], CFG["n_train"] + CFG["n_eval"], 0
for ex in stream:
    scanned += 1
    msgs = ex["messages"]
    if len(msgs) >= 2 and msgs[-1]["role"] == "assistant":
        e = encode(msgs, CFG["seq_len"])
        if e is not None:
            encoded.append(e)
    if len(encoded) >= need or scanned >= 60_000:
        break
assert len(encoded) == need, f"only {len(encoded)} usable rows after {scanned} scanned"
print(f"kept {len(encoded)}/{scanned} scanned rows (length filter <= {CFG['seq_len']} tokens)")

input_ids = torch.full((need, CFG["seq_len"]), PAD_ID, dtype=torch.long)
prompt_lens = []
for i, (full, plen) in enumerate(encoded):
    input_ids[i, :len(full)] = torch.tensor(full)
    prompt_lens.append(plen)
mask = completion_mask_from_prompt_lens(input_ids, prompt_lens, pad_token_id=PAD_ID)
labels = input_ids.clone(); labels[~mask] = -100

# The four-point mask audit. Run these on every dataset you ever build.
assert not (mask & (input_ids == PAD_ID)).any(), "1: no padding is supervised"
frac = mask.float().sum() / (input_ids != PAD_ID).sum()
assert 0.05 < frac < 0.95, f"2: supervised fraction {frac:.2f} is sane"
eos_rows = ((input_ids == EOS_ID) & mask).any(-1).float().mean()
assert eos_rows > 0.99, "3: (almost) every row supervises an EOS"
for i in range(need):
    assert not mask[i, :prompt_lens[i]].any(), "4: no prompt token is supervised"

os.makedirs("../data/lab03", exist_ok=True)
torch.save({"input_ids": input_ids[:CFG["n_train"]], "labels": labels[:CFG["n_train"]],
            "mask": mask[:CFG["n_train"]], "prompt_lens": prompt_lens[:CFG["n_train"]]},
           "../data/lab03/train.pt")
torch.save({"input_ids": input_ids[CFG["n_train"]:], "labels": labels[CFG["n_train"]:],
            "mask": mask[CFG["n_train"]:], "prompt_lens": prompt_lens[CFG["n_train"]:]},
           "../data/lab03/eval.pt")
print(f"train {CFG['n_train']} rows, eval {CFG['n_eval']} rows, seq_len {CFG['seq_len']}, "
      f"supervised frac {frac:.2f} — audit passed, tensors saved")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (14712 > 8192). Running this sequence through the model will result in indexing errors


kept 4352/16112 scanned rows (length filter <= 384 tokens)


train 4096 rows, eval 256 rows, seq_len 384, supervised frac 0.47 — audit passed, tensors saved


## Part B: The run

The trainer below is deliberately *not* a library trainer. Before Lab 07 hands the loop to TRL,
you should have seen every line of a KD step once: student forward, teacher forward under
`no_grad`, one shift applied to **both** logit tensors and the mask together, the Hinton loss,
and nothing else. Note that this whole lab trains *teacher-forced*, which means the student is
always fed the reference text from the corpus and scored on predicting each next token; it
never reads its own generated output during training. The whole method is fifteen lines, and
every distillation bug you will ever debug lives in one of them.

Two engineering notes worth reading even if you never flip the flag:

- The teacher runs under `torch.no_grad()` and in eval mode, and the two guard against
  different bugs. `torch.no_grad()` tells autograd not to record the forward pass; without it,
  autograd saves every intermediate activation in case a backward pass later needs them, which
  roughly doubles the teacher's memory for gradients the teacher will never take. Eval mode
  (`model.eval()`) is separate: PyTorch modules carry a train mode and an eval mode, and in
  train mode dropout layers randomly zero a fraction of activations on every forward pass.
  A teacher accidentally left in train mode therefore emits randomly perturbed logits, which
  means you are distilling a *noised* teacher. That is a real bug, and it presents as a
  mysteriously high floor on the KL, because the student can never match a target that changes
  randomly on every step.
- The eval pass logs the **course diagnostics** (agreement, forward KL, entropy, ECE) rather
  than eval loss. Loss conflates the α-weighted mixture, which means a change in loss could
  have come from either term; the diagnostics separate what actually moved. Two of them need a
  definition. Entropy is the average uncertainty of the student's next-token distribution,
  measured in nats (a nat is the unit of information you get when logarithms are taken in the
  natural base e; a uniform distribution over V tokens has entropy ln V nats, and a fully
  confident distribution has 0). ECE is expected calibration error, and it measures
  *calibration*: whether the model's confidence matches its accuracy. A model that says "80%
  sure" and is right 80% of the time is well calibrated; ECE is the average gap between stated
  confidence and realized accuracy, so lower is better.

In [6]:
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, get_cosine_schedule_with_warmup

def kd_step(student, teacher, batch, alpha, T):
    '''One knowledge-distillation step. This is the entire method.'''
    ids, labels, m = batch["input_ids"], batch["labels"], batch["mask"]
    s_logits = student(ids).logits
    if teacher is not None:
        with torch.no_grad():
            t_logits = teacher(ids).logits
    else:
        t_logits = s_logits.detach()          # unused when alpha == 0
    s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, m)
    return hinton_kd_loss(s_sh, t_sh, labels[:, 1:], m_sh, T=T, alpha=alpha)

@torch.no_grad()
def evaluate(student, teacher, eval_batches):
    '''The diagnostics that matter, on held-out data.'''
    aggs = {"agree": [], "fwd_kl": [], "entropy": [], "ece": []}
    for batch in eval_batches:
        s_logits = student(batch["input_ids"]).logits
        t_logits = teacher(batch["input_ids"]).logits if teacher is not None else s_logits
        s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, batch["mask"])
        aggs["agree"].append(top1_agreement(s_sh, t_sh, m_sh))
        aggs["fwd_kl"].append(float(kl_divergence(s_sh, t_sh, m_sh, scale_by_T2=False)))
        aggs["entropy"].append(mean_entropy(s_sh, m_sh))
        aggs["ece"].append(expected_calibration_error(
            s_sh, batch["input_ids"][:, 1:], m_sh))
    return {k: sum(v) / len(v) for k, v in aggs.items()}

def run_arm(name, cfg, out_root="../runs/lab03"):
    set_seed_everywhere(SEED)
    dtype = getattr(torch, cfg["dtype"])
    student = AutoModelForCausalLM.from_pretrained(cfg["student"], dtype=dtype).to(device)
    teacher = None
    if cfg["teacher"]:
        teacher = AutoModelForCausalLM.from_pretrained(cfg["teacher"], dtype=dtype)
        teacher = teacher.to(device).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)

    tr = torch.load("../data/lab03/train.pt"); ev = torch.load("../data/lab03/eval.pt")
    to_batches = lambda d, bs: [
        {k: d[k][i:i+bs].to(device) for k in ("input_ids", "labels", "mask")}
        for i in range(0, len(d["input_ids"]), bs)]
    train_batches = to_batches(tr, cfg["batch_size"])
    eval_batches = to_batches(ev, cfg["batch_size"])

    opt = torch.optim.AdamW(student.parameters(), lr=cfg["lr"])
    sched = get_cosine_schedule_with_warmup(opt, cfg["warmup_steps"], cfg["max_steps"])
    log, step = [], 0
    while step < cfg["max_steps"]:
        for batch in train_batches:
            loss = kd_step(student, teacher, batch, cfg["alpha"], cfg["T"]) / cfg["grad_accum"]
            loss.backward()
            if (step + 1) % cfg["grad_accum"] == 0:
                torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            if step % 100 == 0:
                metrics = evaluate(student, teacher, eval_batches[:4])
                log.append({"step": step, "loss": float(loss) * cfg["grad_accum"], **metrics})
                print(f"[{name}] step {step:>5}  loss {log[-1]['loss']:.3f}  "
                      f"agree {metrics['agree']:.3f}  KL {metrics['fwd_kl']:.3f}  "
                      f"H {metrics['entropy']:.3f}  ECE {metrics['ece']:.3f}")
            step += 1
            if step >= cfg["max_steps"]:
                break

    fp = config_fingerprint({**cfg, "seed": SEED})
    out = os.path.join(out_root, f"{name}_{fp}")
    os.makedirs(out, exist_ok=True)
    student.save_pretrained(out)
    json.dump(log, open(os.path.join(out, "log.json"), "w"), indent=2)
    RunManifest(name=name, config=cfg, seed=SEED,
                artifacts_in={"train": "lab03/train.pt"},
                artifacts_out={"checkpoint": out}).save(out)
    del student, teacher
    if device == "cuda":
        torch.cuda.empty_cache()
    return out

if RUN_TRAINING:
    results = {name: run_arm(name, cfg) for name, cfg in ARMS.items()}
    print(json.dumps(results, indent=2))
else:
    print("RUN_TRAINING=False — Part B compiled but did not execute.")
    print("On the training box: flip the flag, rerun this cell, ~30-60 min per arm.")

RUN_TRAINING=False — Part B compiled but did not execute.
On the training box: flip the flag, rerun this cell, ~30-60 min per arm.


## Part C: The verdict

**Expected ranges** (from published SmolLM2-scale KD runs and the capacity-gap literature; your
numbers will differ, your *orderings* mostly should not):

- `soft` and `mixed` beat `hard` on top-1 agreement with the 1.7B teacher by **2–8 points**
  at equal steps, and on ECE by a visible margin. If soft targets do not beat hard labels here,
  with an in-family teacher, in-distribution data, and a healthy teacher, something is wrong
  with the setup, not the theory.
- `mixed` vs `soft` is genuinely close; α=0.5's advantage shows up more on ECE than on
  agreement, because the hard term anchors calibration: it keeps pulling probability onto
  tokens that are actually correct, which keeps the student's confidence tied to its accuracy.
  Either ordering is a legitimate result.
- The gap probe: `gap-large` (12.6× teacher) should beat `gap-small` (2.7×) *by less than the
  teacher-size ratio suggests*, may tie it, and a small loss for `gap-large` is a real,
  publishable-grade observation. That is the capacity gap showing up in your own logs. What
  would be surprising is a large `gap-large` win.

**Failure signatures**, from most to least likely:

- *KL flat from step 0, agreement at the random-init level (~0).* The mask or shift is wrong,
  which means you are training on prompt tokens or on unshifted positions. Part A·4's audit
  passed, so suspect device moves or a re-tokenization that happened between the audit and
  training.
- *Loss falls, agreement rises, ECE rises too.* The student is getting confident faster than it
  is getting right. Lower α toward the hard term, or lower T.
- *Soft arms no better than `hard`.* Check the teacher is in eval mode and under `no_grad`
  (a dropout-noised teacher's dark knowledge is mostly noise), and check `T`: at T=1 with a
  very peaked teacher, soft targets *are* nearly hard labels, because almost all the
  probability sits on one token and the wrong-answer structure is too small to teach anything.
- *Early NaN.* Almost always the fp16-style pathology from Lab 00 §7. Confirm the loss math
  runs in fp32 even though the models are bf16 (`kd_core` upcasts internally via log_softmax on
  fp32 tensors when given float logits; keep it that way).

**Write the verdict.** Three sentences, before looking at anything else: Did soft targets beat
hard labels, on which diagnostic, by how much? Did the bigger teacher earn its memory? What
would you change in the next run? If you cannot write these from your `log.json` files in two
minutes, the run was not instrumented well enough, and that is itself the finding.

## Exercises

1. **Sweep T ∈ {1, 2, 4, 8} on the `soft` arm.** Hinton's paper says intermediate temperatures
   win. Verify, and connect what you see at T=8 to Lab 01 §3 (KD → logit MSE at high T).
2. **Give the teacher label smoothing.** Fine-tune the 1.7B teacher briefly with label smoothing
   0.1, re-run `soft`, and watch the student get *worse* despite an unchanged-or-better teacher
   (Müller et al.'s result). Explain via what smoothing does to the wrong-answer probabilities.
3. **Length-stratify the eval.** Split eval rows by completion length and recompute agreement.
   KD's advantage typically concentrates on longer completions. Why?
4. **The patient teacher.** Double `max_steps` on `gap-large` only. Beyer et al. predict the
   capacity gap partially closes with patience. Does it?